# Creating Custom Function Nodes

In this tutorial we do a deep dive into function nodes and how users can create their own nodes for various functions.  Users should already be familiar with the concepts covered in the [Sampling Parameters notebook](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/sampling.html).

In [ ]:
from lightcurvelynx.base_models import FunctionNode
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc

## Function Nodes

Function nodes provide users the ability to wrap arbitrary computation functions in parameter generation. There are two ways to use the `FunctionNode` class: as a standalone wrapper or as a parent class.


### FunctionNode as a Standalone Wrapper

Users can wrap a function directly by passing the function and its arguments into the `FunctionNode` constructor. This wraps the provided function and uses the functions returned value as its output.

As a concrete example, let's create a `FunctionNode` that computes wraps an existing function that computes y = m * x + b. We need to pass this function and values for each of its parameters to the constructor. Like general `ParameterizedNode` objects, the parameter values can be set as constants, parameter values from other nodes, or results from other function nodes.

In [ ]:
# This is the function we would like to wrap.
def linear_eq_function(x, m, b):
    """Compute y = m * x + b"""
    return m * x + b


# This is how we wrap linear_eq_function.
func_node = FunctionNode(
    linear_eq_function,  # First argument is the function to call.
    # The next arguments are the function's parameters.
    x=NumpyRandomFunc("uniform", low=0.0, high=10.0),  # Random value
    m=5.0,  # Constant value
    b=-2.0,  # Constant value
)

The `FunctionNode` class handles all the internal book keeping of determining the names of the function's arguments and assembling those arguments whenever the function is called.

When we run the sampling and print out `GraphState` we can see the different x values sampled by the `NumpyRandomFunc` and fed into the `x` parameter. You can see all of the internal state and book keeping parameters.

In [ ]:
state = func_node.sample_parameters(num_samples=5)
print(state)

More realistically, users can wrap functions that perform complex astronomical calculations. However, the downside of the basic wrapper approach is that the node itself does not store any additional state.

### FunctionNode Subclasses



In [ ]:
from astropy.cosmology import FlatLambdaCDM


class DistModFromRedshift(FunctionNode):
    """A wrapper class for the _distmod_from_redshift() function.

    Parameters
    ----------
    redshift : function or constant
        The function or constant providing the redshift value.
    H0 : constant
        The Hubble constant.
    Omega_m : constant
        The matter density Omega_m.
    **kwargs : dict, optional
        Any additional keyword arguments.
    """

    def __init__(self, redshift, H0=73.0, Omega_m=0.3, **kwargs):
        # Create the cosmology once for this node.
        if not isinstance(H0, float) or not isinstance(Omega_m, float):
            raise ValueError("H0 and Omega_m must be constants.")
        self.cosmo = FlatLambdaCDM(H0=H0, Om0=Omega_m)

        # Call the super class's constructor with the needed information.
        super().__init__(
            func=self._distmod_from_redshift,
            redshift=redshift,
            **kwargs,
        )

    def _distmod_from_redshift(self, redshift):
        """Compute distance modulus given redshift and cosmology.

        Parameters
        ----------
        redshift : float or numpy.ndarray
            The redshift value(s).

        Returns
        -------
        distmod : float or numpy.ndarray
            The distance modulus (in mag)
        """
        return self.cosmo.distmod(redshift).value

### Creating New Function Nodes

Importantly, you can use a `FunctionNode` that takes input parameter(s) and produces output parameter(s) during the generation. `FunctionNode` is a subclass of `ParameterizedNode` that wraps the functionality of collecting the inputs, running the computations, and storing the output.

As a simple example, let's create a `FunctionNode` that computes y = m * x + b.

In [ ]:
from lightcurvelynx.base_models import FunctionNode


def _linear_eq(x, m, b):
    """Compute y = m * x + b"""
    return m * x + b


func_node = FunctionNode(
    _linear_eq,  # First parameter is the function to call.
    x=NumpyRandomFunc("uniform", low=0.0, high=10.0),
    m=5.0,
    b=-2.0,
)
print(func_node.sample_parameters(num_samples=1))

The first parameter of the function node is the function to evaluate, such as our linear equation above. Each input into that function **must** be included as a named parameter during the `FunctionNode` definition, such as `x`, `m`, and `b` above. If any of the input parameters are missing the code will give an error.

Here we use constants for `m` and `b` so we use the same linear formulation for each sample. Only the value of `x` changes. However we could have also used function nodes, including sampling functions, to set `m` and `b`. In that case it is important to remember that each of our results sample will be the result of a sampling of all the variable parameters.